# RBF support vector machine — daily flood occurrence

Trains `SVC(kernel="rbf")` on `dataset/flood_training_data_split.csv` and reports
accuracy, precision, recall, F1 and MCC on the held-out test period.

| Stage | Detail |
| --- | --- |
| Features | `ante_15d` (log1p + standardised), `tmin_c` (standardised) |
| Split | chronological, pre-built: train to 2014-02-28, test from 2014-03-01 |
| Imbalance | `class_weight="balanced"` plus negatives subsampled 50:1 |
| Model | RBF SVC, averaged over 15 negative subsamples |
| Threshold | F1-optimal on the last 20% of train |

Three choices specific to this model:

- **Scaling is mandatory.** The RBF kernel is a distance function, and `ante_15d`
  spans 5–271 mm against `tmin_c`'s 9–21 °C. Unscaled, rainfall dominates it
  entirely.
- **`decision_function`, not `probability=True`.** Platt scaling refits the model
  five times internally, and only a ranking is needed to pick a threshold.
- **Negatives subsampled and the fit repeated.** `SVC` is roughly O(n²)–O(n³), so
  38,799 rows is impractical; and a single subsample is not a result, because
  test PR-AUC swings 2.7-fold across draws. Fifteen fits are averaged.

Two features, not nine: cross-validated ROC-AUC fell monotonically as features
were added (0.714 on these two, 0.597 on all nine), because `district` and
day-of-year encode reporting patterns that reverse between the two periods.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             matthews_corrcoef, precision_recall_curve,
                             precision_score, recall_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import SVC

LABEL = "Flood occurrences"
NEGATIVE_RATIO = 50   # non-floods kept per flood when fitting
N_SUBSAMPLES = 15     # fits averaged, to remove the luck of a single draw


def find_repo_root(marker: str = "dataset") -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise FileNotFoundError(f"No parent of {Path.cwd()} contains {marker!r}")


REPO_ROOT = find_repo_root()
data = pd.read_csv(REPO_ROOT / "dataset/flood_training_data_split.csv", parse_dates=["date"])
data = data.sort_values("date").reset_index(drop=True)
data[LABEL] = data[LABEL].astype(bool)

train = data[data["split"] == "train"]
test = data[data["split"] == "test"]
y_train = train[LABEL].to_numpy()
y_test = test[LABEL].to_numpy()

print(f"train {len(train):,} rows, {y_train.sum()} floods   "
      f"test {len(test):,} rows, {y_test.sum()} floods")

In [ ]:
def build_model() -> Pipeline:
    """Scaled features, RBF SVC with balanced class weights.

    Scaling sits inside the pipeline so it is refitted on each subsample rather
    than once across data the model has not seen. gamma="scale" resolves to
    1 / (n_features * X.var()); inputs are standardised so for two features this
    is ~0.5, and setting gamma=0.5 explicitly gives the identical model.
    """
    return Pipeline([
        ("prep", ColumnTransformer([
            ("rain", Pipeline([("log", FunctionTransformer(np.log1p)),
                               ("scale", StandardScaler())]), ["ante_15d"]),
            ("temp", StandardScaler(), ["tmin_c"]),
        ])),
        ("model", SVC(kernel="rbf", class_weight="balanced", gamma="scale", cache_size=800)),
    ])


def subsample_negatives(frame, labels, seed):
    """Keep every flood, sample NEGATIVE_RATIO non-floods per flood."""
    rng = np.random.default_rng(seed)
    positives = np.flatnonzero(labels)
    negatives = np.flatnonzero(~labels)
    take = min(len(negatives), NEGATIVE_RATIO * len(positives))
    keep = np.sort(np.concatenate([positives, rng.choice(negatives, take, replace=False)]))
    return frame.iloc[keep], labels[keep]


def fit_and_score(fit_frame, fit_labels, score_frame):
    """Average the decision function over N_SUBSAMPLES fits.

    Raw margins are averaged rather than per-model standardised ones, so the
    output stays on one fixed scale - which is what lets a threshold chosen on
    validation transfer to test.
    """
    total = np.zeros(len(score_frame))
    for seed in range(N_SUBSAMPLES):
        subset, subset_labels = subsample_negatives(fit_frame, fit_labels, seed)
        total += build_model().fit(subset, subset_labels).decision_function(score_frame)
    return total / N_SUBSAMPLES


# Threshold picked on the last 20% of train; test is not consulted
cut = int(len(train) * 0.8)
validation_scores = fit_and_score(train.iloc[:cut], y_train[:cut], train.iloc[cut:])

precision, recall, thresholds = precision_recall_curve(y_train[cut:], validation_scores)
f1_curve = np.divide(2 * precision * recall, precision + recall,
                     out=np.zeros_like(precision), where=(precision + recall) > 0)
# precision_recall_curve returns one more point than it does thresholds
THRESHOLD = float(thresholds[max(0, int(np.argmax(f1_curve)) - 1)])

# Refit on the full training split, score test once
test_scores = fit_and_score(train, y_train, test)
predictions = test_scores >= THRESHOLD

print(f"threshold {THRESHOLD:.4f}   flagged {predictions.sum():,} of {len(test):,} test days")

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, predictions).ravel()

print(f"TP {tp}   FP {fp:,}   FN {fn}   TN {tn:,}\n")
for name, value in {
    "Accuracy": accuracy_score(y_test, predictions),
    "Precision": precision_score(y_test, predictions, zero_division=0),
    "Recall": recall_score(y_test, predictions),
    "F1 Score": f1_score(y_test, predictions),
    "MCC": matthews_corrcoef(y_test, predictions),
}.items():
    print(f"{name:<10} {value:.4f}")

## Results

| Metric | RBF SVM | Naive Bayes | KNN |
| --- | --- | --- | --- |
| Accuracy | 0.9090 | 0.8198 | 0.7385 |
| Precision | 0.0046 | 0.0063 | 0.0059 |
| Recall | 0.1053 | 0.2895 | 0.3947 |
| F1 Score | 0.0089 | 0.0123 | 0.0116 |
| **MCC** | **0.0038** | 0.0181 | 0.0191 |
| PR-AUC | **0.0120** | 0.0099 | 0.0054 |

Confusion matrix: TP 4, FP 858, FN 34, TN 8,910.

**The SVM has the worst MCC of the three and the best ranking.** MCC 0.0038
against 0.018–0.019 for the other two, yet PR-AUC 0.0120 is the highest — it
orders the test days better than either alternative while its threshold lands
badly. It flags only 862 days and catches 4 floods of 38, where KNN flags 2,556
and catches 15.

**A single subsample would not have been a result.** Across the 15 draws, test
PR-AUC ranges 0.0068 to 0.0187 (mean 0.0119, sd 0.0041) — a 2.7-fold spread caused
only by which negatives were sampled. With 38 test floods, PR-AUC is dominated by
a handful of top-ranked rows. An earlier draft of this notebook reported 0.0318
from one seed, the maximum of twelve tried, which would have overstated
performance by 2.6x.

**Accuracy of 0.9090 is the highest of the three models and the least
informative.** Predicting "no flood" for every row scores 0.9961 at MCC exactly 0,
so the SVM's apparent advantage on accuracy is only its reluctance to flag
anything.

In absolute terms all three models sit close to chance. The limit is shared: 83
training floods, a best single-feature ROC-AUC of 0.694, and a label process whose
seasonal and spatial patterns reverse between the two periods.